In [2]:
import pygame
from collections import deque
import heapq

# ---------- Labyrinthe ----------
maze = [
    [1,1,1,1,1,1,1,1,1,1,1,1],
    [1,'S',0,0,0,1,0,0,0,0,0,1],
    [1,0,1,0,1,1,0,1,1,1,0,1],
    [1,0,1,0,0,0,0,0,0,1,0,1],
    [1,0,1,1,1,0,1,1,0,1,0,1],
    [1,0,0,0,1,0,1,0,0,1,0,1],
    [1,1,1,0,1,0,1,0,1,1,0,1],
    [1,0,0,0,0,0,0,0,1,0,0,1],
    [1,0,1,1,1,1,1,0,1,0,1,1],
    [1,0,0,0,0,0,0,0,0,0,'E',1],
    [1,1,1,1,1,1,1,1,1,1,1,1]
]

CELL_SIZE = 40
ROWS = len(maze)
COLS = len(maze[0])
WIDTH = COLS * CELL_SIZE
HEIGHT = ROWS * CELL_SIZE + 60  # espace pour les boutons

# ---------- Pygame setup ----------
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Maze Solver Animé")
clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 30)

# ---------- Couleurs ----------
BLACK   = (0,0,0)
WHITE   = (255,255,255)
GREEN   = (0,255,0)
RED     = (255,0,0)
BLUE    = (0,0,255)
CYAN    = (0,255,255)
YELLOW  = (255,255,0)
ORANGE  = (255,165,0)
MAGENTA = (255,0,255)
GRAY    = (100,100,100)

DFS_COLOR = (0,255,128)
BFS_COLOR = (0,128,128)
BFS_FINAL_COLOR = (0,255,255)
ASTAR_COLOR = (255,0,255)
ASTAR_FINAL_COLOR = (255,165,0)
WF_COLOR = (255,128,0)

# ---------- Départ et fin ----------
for i,row in enumerate(maze):
    for j,cell in enumerate(row):
        if cell=='S': start_pos = (i,j)
        if cell=='E': end_pos = (i,j)
robot_pos = list(start_pos)

# ---------- Variables algos ----------
dfs_stack = []; dfs_visited = set(); dfs_path_cells=set()
bfs_queue = deque(); bfs_visited=set(); bfs_path={}; bfs_path_cells=set(); bfs_final_path=set()
a_heap=[]; a_visited=set(); a_path={}; a_star_path_cells=set(); a_star_final_path=set()
wf_x, wf_y = start_pos; wf_dx, wf_dy = 0,1; wf_path_cells=set()
algorithm = None

# ---------- Boutons ----------
buttons = [
    {"text":"DFS", "rect":pygame.Rect(10, HEIGHT-50, 80, 40), "action":"DFS"},
    {"text":"BFS", "rect":pygame.Rect(100, HEIGHT-50, 80, 40), "action":"BFS"},
    {"text":"A*", "rect":pygame.Rect(190, HEIGHT-50, 80, 40), "action":"A*"},
    {"text":"Wall Follower", "rect":pygame.Rect(280, HEIGHT-50, 140, 40), "action":"WF"}
]

def draw_buttons():
    for b in buttons:
        pygame.draw.rect(screen, WHITE, b["rect"])
        pygame.draw.rect(screen, BLACK, b["rect"], 2)
        text_surf = font.render(b["text"], True, BLACK)
        text_rect = text_surf.get_rect(center=b["rect"].center)
        screen.blit(text_surf, text_rect)

# ---------- Dessin ----------
def draw_maze():
    screen.fill(BLACK)
    for i in range(ROWS):
        for j in range(COLS):
            x, y = j*CELL_SIZE, i*CELL_SIZE
            cell = maze[i][j]
            if cell==1: color=BLACK
            elif cell==0: color=WHITE
            elif cell=='S': color=GREEN
            elif cell=='E': color=RED
            pygame.draw.rect(screen,color,(x,y,CELL_SIZE,CELL_SIZE))
            pygame.draw.rect(screen,GRAY,(x,y,CELL_SIZE,CELL_SIZE),1)

    # Chemins explorés
    for cx,cy in dfs_path_cells: pygame.draw.rect(screen,DFS_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))
    for cx,cy in bfs_path_cells: pygame.draw.rect(screen,BFS_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))
    for cx,cy in a_star_path_cells: pygame.draw.rect(screen,ASTAR_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))
    for cx,cy in wf_path_cells: pygame.draw.rect(screen,WF_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))

    # Chemins finaux BFS / A*
    for cx,cy in bfs_final_path: pygame.draw.rect(screen,BFS_FINAL_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))
    for cx,cy in a_star_final_path: pygame.draw.rect(screen,ASTAR_FINAL_COLOR,(cy*CELL_SIZE,cx*CELL_SIZE,CELL_SIZE,CELL_SIZE))

    # Robot
    rx, ry = robot_pos
    pygame.draw.rect(screen, BLUE, (ry*CELL_SIZE, rx*CELL_SIZE, CELL_SIZE, CELL_SIZE))

    draw_buttons()

# ---------- Reset chemins ----------
def reset_paths():
    global dfs_path_cells, bfs_path_cells, a_star_path_cells, wf_path_cells
    global robot_pos, algorithm, bfs_final_path, a_star_final_path, dfs_stack, bfs_queue, a_heap
    dfs_path_cells = set()
    bfs_path_cells = set()
    a_star_path_cells = set()
    wf_path_cells = set()
    bfs_final_path = set()
    a_star_final_path = set()
    dfs_stack = []
    bfs_queue = deque()
    a_heap = []
    robot_pos[:] = list(start_pos)
    algorithm = None

# ---------- DFS ----------
def dfs_step():
    global robot_pos
    if not dfs_stack: return
    x,y = dfs_stack.pop()
    if (x,y) in dfs_visited or maze[x][y]==1: return
    dfs_visited.add((x,y))
    dfs_path_cells.add((x,y))
    robot_pos[:] = [x,y]
    if maze[x][y]=='E':
        dfs_stack.clear()
        print("DFS terminé !")
        return
    for dx,dy in [(0,1),(0,-1),(1,0),(-1,0)]:
        nx,ny=x+dx,y+dy
        if 0<=nx<ROWS and 0<=ny<COLS and (nx,ny) not in dfs_visited and maze[nx][ny]!=1:
            dfs_stack.append((nx,ny))

def run_dfs():
    reset_paths()
    global dfs_stack, dfs_visited, dfs_path_cells, robot_pos, algorithm
    dfs_stack=[start_pos]; dfs_visited=set(); dfs_path_cells=set(); robot_pos[:] = list(start_pos)
    algorithm="DFS"

# ---------- BFS ----------
def bfs_step():
    global robot_pos, bfs_final_path
    if not bfs_queue: return
    x,y = bfs_queue.popleft()
    bfs_path_cells.add((x,y))
    robot_pos[:] = [x,y]
    if (x,y)==end_pos:
        bfs_queue.clear()
        print("BFS terminé !")
        # Calculer path optimale
        bfs_final_path = set()
        p = (x,y)
        while p != start_pos:
            bfs_final_path.add(p)
            p = bfs_path[p]
        bfs_final_path.add(start_pos)
        return
    for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
        nx,ny=x+dx,y+dy
        if 0<=nx<ROWS and 0<=ny<COLS and maze[nx][ny]!=1 and (nx,ny) not in bfs_visited:
            bfs_visited.add((nx,ny))
            bfs_path[(nx,ny)] = (x,y)
            bfs_queue.append((nx,ny))

def run_bfs():
    reset_paths()
    global bfs_queue, bfs_visited, bfs_path, bfs_path_cells, bfs_final_path, robot_pos, algorithm
    bfs_queue=deque([start_pos]); bfs_visited=set([start_pos]); bfs_path={}; bfs_path_cells=set()
    bfs_final_path=set(); robot_pos[:] = list(start_pos)
    algorithm="BFS"

# ---------- A* ----------
def heuristic(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])

def a_star_step():
    global robot_pos, a_star_final_path
    if not a_heap: return
    _,(x,y),cost = heapq.heappop(a_heap)
    if (x,y) in a_visited: return
    a_visited.add((x,y))
    a_star_path_cells.add((x,y))
    robot_pos[:] = [x,y]
    if (x,y)==end_pos:
        a_heap.clear()
        print("A* terminé !")
        # Calculer path optimale
        a_star_final_path = set()
        p = (x,y)
        while p != start_pos:
            a_star_final_path.add(p)
            p = a_path[p]
        a_star_final_path.add(start_pos)
        return
    for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
        nx,ny=x+dx,y+dy
        if 0<=nx<ROWS and 0<=ny<COLS and maze[nx][ny]!=1 and (nx,ny) not in a_visited:
            heapq.heappush(a_heap,(cost+1+heuristic((nx,ny),end_pos),(nx,ny),cost+1))
            a_path[(nx,ny)] = (x,y)

def run_a_star():
    reset_paths()
    global a_heap, a_visited, a_path, a_star_path_cells, a_star_final_path, robot_pos, algorithm
    a_heap=[(0,start_pos,0)]; a_visited=set(); a_path={}; a_star_path_cells=set()
    a_star_final_path=set(); robot_pos[:] = list(start_pos)
    algorithm="A*"

# ---------- Wall Follower ----------
def wf_step():
    global wf_x, wf_y, wf_dx, wf_dy, robot_pos, algorithm
    robot_pos[:] = [wf_x,wf_y]
    wf_path_cells.add((wf_x,wf_y))
    if maze[wf_x][wf_y]=='E':
        print("Wall Follower terminé !")
        algorithm=None
        return

    fx, fy = wf_x+wf_dx, wf_y+wf_dy
    rx, ry = wf_x-wf_dy, wf_y+wf_dx
    lx, ly = wf_x+wf_dy, wf_y-wf_dx

    if 0<=rx<ROWS and 0<=ry<COLS and maze[rx][ry]!=1:
        wf_dx, wf_dy = -wf_dy, wf_dx
        wf_x += wf_dx; wf_y += wf_dy
    elif 0<=fx<ROWS and 0<=fy<COLS and maze[fx][fy]!=1:
        wf_x += wf_dx; wf_y += wf_dy
    elif 0<=lx<ROWS and 0<=ly<COLS and maze[lx][ly]!=1:
        wf_dx, wf_dy = wf_dy, -wf_dx
        wf_x += wf_dx; wf_y += wf_dy
    else:
        wf_dx, wf_dy = -wf_dx, -wf_dy
        wf_x += wf_dx; wf_y += wf_dy

def run_wf():
    reset_paths()
    global wf_x, wf_y, wf_dx, wf_dy, wf_path_cells, robot_pos, algorithm
    wf_x, wf_y = start_pos; wf_dx, wf_dy=0,1; wf_path_cells=set(); robot_pos[:] = list(start_pos)
    algorithm="WF"

# ---------- Boucle principale ----------
running = True
while running:
    clock.tick(30)
    draw_maze()
    pygame.display.flip()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = event.pos
            for b in buttons:
                if b["rect"].collidepoint(mx, my):
                    if b["action"] == "DFS": run_dfs()
                    elif b["action"] == "BFS": run_bfs()
                    elif b["action"] == "A*": run_a_star()
                    elif b["action"] == "WF": run_wf()

    # Mise à jour robot
    if algorithm == "DFS": dfs_step()
    elif algorithm == "BFS": bfs_step()
    elif algorithm == "A*": a_star_step()
    elif algorithm == "WF": wf_step()

pygame.quit()
